# معمل بناء أول وكيل ذكاء اصطناعي موحد عبر `create_agent`

أهلاً بك في المعمل العملي للمحاضرة 59!

في المعمل ده هنجرب عملياً إزاي نبني أول AI Agent في LangChain v1+ باستخدام دالة `create_agent` الموحدة:
1. نشوف الفرق بين وكيل من غير أدوات ووكيل متوصل بأداة الطقس.
2. نفهم إزاي الموديل بيقرا الـ `Docstring` بتاع الدالة عشان يعرف يختارها.
3. نستدعي الوكيل ونشوف رحلة الرسائل الأربعة لحد ما يرجع لنا الإجابة!


In [1]:
# تجهيز البيئة وتحميل الموديل
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq

load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("تم تحميل الموديل بنجاح وجاهزين نبني الوكيل.")


تم تحميل الموديل بنجاح وجاهزين نبني الوكيل.


### 1. بناء وكيل بدون أدوات (Tool-less Agent)

تعال نجرب نبني وكيل بـ `tools=[]` ونشوف شكل الرسم البياني بتاعه جوا LangGraph.


In [2]:
basic_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful assistant."
)

print("عقد الوكيل بدون أدوات:", list(basic_agent.get_graph().nodes.keys()))


عقد الوكيل بدون أدوات: ['__start__', 'model', '__end__']


### 2. تعريف أداة الطقس وربطها بالوكيل

لاحظ هنا إن السطر المكتوب جوا `"""..."""` (الـ Docstring) هو اللي بيعرف الـ LLM إن الدالة دي مسؤولة عن الطقس!


In [3]:
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny."

weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant."
)

print("عقد الوكيل بعد إضافة الأداة:", list(weather_agent.get_graph().nodes.keys()))


عقد الوكيل بعد إضافة الأداة: ['__start__', 'model', 'tools', '__end__']


### 3. تشغيل الوكيل ومراقبة رحلة الرسائل

دلوقتي هنسأل الوكيل عن طقس نيويورك، وهنطبع كل خطوة عملها (سؤالنا -> قرار استدعاء الأداة -> نتيجة الأداة -> الرد النهائي).


In [4]:
response = weather_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather in New York?"}]
})

for i, msg in enumerate(response["messages"], 1):
    msg_type = type(msg).__name__
    print(f"خطوة {i} [{msg_type}]: {msg.content or getattr(msg, 'tool_calls', '')}")

print("\nالرد النهائي المختصر:", response["messages"][-1].content)


خطوة 1 [HumanMessage]: What is the weather in New York?
خطوة 2 [AIMessage]: [{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'fc_b79ab765-2d0b-425a-92b4-5824cea70164', 'type': 'tool_call'}]
خطوة 3 [ToolMessage]: The weather in New York is sunny.
خطوة 4 [AIMessage]: The weather in New York is sunny.

الرد النهائي المختصر: The weather in New York is sunny.


### 4. جرب بنفسك!

دلوقتي دورك في التجربة الحرة:
1. جرب تبعت السؤال بصيغة مختصرة وحتى بغلط إملائي زي ما شفنا في الفيديو (`"What is the wather in Cairo?"`).
2. جرب تضيف أداة تانية بسيطة (مثلاً آلة حاسبة صغيرة أو معرفة التوقيت) وتضيفها لقائمة `tools=[get_weather, ...]` وتشوف الوكيل هيختار الأداة الصح إزاي لوحده!


In [5]:
# جرب بنفسك هنا مع الصيغة المختصرة أو خطأ إملائي بسيط
my_test = weather_agent.invoke({"messages": "What is the wather in Cairo?"})

print("رد الوكيل على سؤالك:", my_test["messages"][-1].content)


رد الوكيل على سؤالك: The weather in Cairo is sunny.
